In [1]:
import pickle
import numpy as np
import pandas as pd
from enums import Frequency # import necessary for loading pickle file
from sklearn.model_selection import TimeSeriesSplit
from model_wrappers import wrapper_abstract, ets_wrapper
from model_wrappers.ets_wrapper import ExponentialSmoothingWrapper
from model_wrappers.autoets_wrapper import AutoETSWrapper

In [2]:
# load categorized timeseries data
with open('categorized_timeseries_dict.pkl', 'rb') as f:
    ts_all = pickle.load(f)

In [3]:
def get_num_of_splits(ts_length:int, ts_frequency:Frequency) -> int:
    # TODO: implement sensible split number logic based on timeseries length and sampling frequency
    return 5


In [18]:
def cross_validate_models(models:list, ts_dict:dict, use_boxcox:bool = True):
    for ts_name in ts_dict.keys():
        ts = ts_dict[ts_name]
        df = pd.DataFrame(ts['original'])
        df_boxcox = pd.DataFrame(ts['boxcox'])
        frequency = ts['frequency']
        values = df_boxcox.values if use_boxcox else df.values

        n_splits = get_num_of_splits(df.size, frequency)
        tscv = frequency.get_tscv(n_splits)
        period = frequency.get_period()

        model_squared_errors = [[] for _ in models]
        for train, test in tscv.split(values):
            model_index = 0
            for model in models:
                model.fit(data=values[train], seasonal_periods=period)
                forecast = model.forecast(steps=frequency.get_forecasting_horizon())
                target = values[test]
                error = np.sum((target - forecast)**2)
                model_squared_errors[model_index].append(error)
                model_index += 1
        mse = [np.mean(squared_error) for squared_error in model_squared_errors]
        print(np.argmin(mse))

ets = [
    ExponentialSmoothingWrapper(
        seasonal="multiplicative",
        damped_trend=False,
        trend="additive",
        initialization_method="estimated",),
    ExponentialSmoothingWrapper(
        seasonal="multiplicative",
        damped_trend=True,
        trend="additive",
        initialization_method="estimated",),
    ExponentialSmoothingWrapper(
        seasonal="additive",
        trend="additive",
        damped_trend=False,
        initialization_method="estimated",),
    ExponentialSmoothingWrapper(
        seasonal="additive",
        damped_trend=True,
        trend="additive",
        initialization_method="estimated",),
    ExponentialSmoothingWrapper(
        seasonal=None,
        damped_trend=True,
        trend="additive",
        initialization_method="estimated",),
    ExponentialSmoothingWrapper(
        seasonal=None,
        damped_trend=False,
        trend="additive",
        initialization_method="estimated",),
    ExponentialSmoothingWrapper(
        seasonal=None,
        damped_trend=False,
        trend=None,
        initialization_method="estimated",),
    ExponentialSmoothingWrapper(
        seasonal="multiplicative",
        damped_trend=False,
        trend=None,
        initialization_method="estimated",),
    ExponentialSmoothingWrapper(
        seasonal="additive",
        damped_trend=False,
        trend=None,
        initialization_method="estimated",)
]
#autoets_test = AutoETSWrapper(auto=True, allow_multiplicative_trend=True, n_jobs=-1)
cross_validate_models(ets, ts_all)

2
2


KeyboardInterrupt: 